**Setup ambiente**

In [22]:
!pip -q install -U transformers datasets evaluate accelerate scikit-learn

import os, json, numpy as np
import torch
from datasets import load_dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report


**Path + LOAD DATASET (HF datasets) + LABEL MAP**

In [ ]:
import os, json
import pandas as pd
from datasets import load_dataset, DatasetDict

BASE = "/content/Semantic"
IN_PATH = os.path.join(BASE, "dataset_en.json")
OUT_DIR = os.path.join(BASE, "dataset_002")
os.makedirs(OUT_DIR, exist_ok=True)

train_clf_path = os.path.join(OUT_DIR, "train_classification.json")
test_clf_path  = os.path.join(OUT_DIR, "test_classification.json")

# --- build dataset_002 se manca ---
if not (os.path.exists(train_clf_path) and os.path.exists(test_clf_path)):

    raw_hf = load_dataset("json", data_files=IN_PATH)["train"]

    # split 80/20 SENZA sklearn
    split_pairs = raw_hf.train_test_split(test_size=0.20, seed=42, shuffle=True)
    train_pairs = split_pairs["train"].to_pandas()
    test_pairs  = split_pairs["test"].to_pandas()

    # normalizza nomi colonna
    train_pairs = train_pairs.rename(columns={"non_inclusiva":"not_inclusive", "inclusiva":"inclusive"})[["not_inclusive","inclusive"]].dropna()
    test_pairs  = test_pairs.rename(columns={"non_inclusiva":"not_inclusive", "inclusiva":"inclusive"})[["not_inclusive","inclusive"]].dropna()

    # pulizia base
    for df_ in (train_pairs, test_pairs):
        df_["not_inclusive"] = df_["not_inclusive"].astype(str).str.strip()
        df_["inclusive"] = df_["inclusive"].astype(str).str.strip()
    train_pairs = train_pairs[(train_pairs["not_inclusive"]!="") & (train_pairs["inclusive"]!="")].reset_index(drop=True)
    test_pairs  = test_pairs[(test_pairs["not_inclusive"]!="") & (test_pairs["inclusive"]!="")].reset_index(drop=True)

    def make_classification(df_):
        rows = []
        for _, r in df_.iterrows():
            rows.append({"text": r["inclusive"], "label": "inclusive"})
            rows.append({"text": r["not_inclusive"], "label": "not_inclusive"})
        return pd.DataFrame(rows)

    train_df = make_classification(train_pairs).sample(frac=1, random_state=42).reset_index(drop=True)
    test_df  = make_classification(test_pairs).sample(frac=1, random_state=42).reset_index(drop=True)

    def save_jsonl(df_, path):
        with open(path, "w", encoding="utf-8") as f:
            for _, r in df_.iterrows():
                f.write(json.dumps(r.to_dict(), ensure_ascii=False) + "\n")

    save_jsonl(train_df, train_clf_path)
    save_jsonl(test_df,  test_clf_path)

    print("Creati:", train_clf_path, "e", test_clf_path)

# --- load HF datasets ---
ds = load_dataset("json", data_files={"train": train_clf_path, "test": test_clf_path})

# validation dal train (10%)
split = ds["train"].train_test_split(test_size=0.1, seed=42, shuffle=True)
ds = DatasetDict({"train": split["train"], "validation": split["test"], "test": ds["test"]})

print(ds)
print("Esempio:", ds["train"][0])

# --- label map ---
labels = sorted(list(set(ds["train"]["label"])))
LABEL2ID = {lab:i for i, lab in enumerate(labels)}
ID2LABEL = {i:lab for lab,i in LABEL2ID.items()}
print("LABEL2ID:", LABEL2ID)

def encode_label(ex):
    ex["label_id"] = LABEL2ID[ex["label"]]
    return ex

ds = ds.map(encode_label)


✅ Creati: /content/Semantic/dataset_002/train_classification.json e /content/Semantic/dataset_002/test_classification.json


Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 7171
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 797
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1992
    })
})
Esempio: {'text': "She's a single person over 30, she must be desperate.", 'label': 'not_inclusive'}
LABEL2ID: {'inclusive': 0, 'not_inclusive': 1}


Map:   0%|          | 0/7171 [00:00<?, ? examples/s]

Map:   0%|          | 0/797 [00:00<?, ? examples/s]

Map:   0%|          | 0/1992 [00:00<?, ? examples/s]

**Check leakage**

In [24]:
train_texts = set(ds["train"]["text"])
val_texts   = set(ds["validation"]["text"])
test_texts  = set(ds["test"]["text"])

print("Overlap train∩val :", len(train_texts & val_texts))
print("Overlap train∩test:", len(train_texts & test_texts))
print("Overlap val∩test  :", len(val_texts & test_texts))


Overlap train∩val : 6
Overlap train∩test: 19
Overlap val∩test  : 0


**TOKENIZE**

In [ ]:
MODEL_NAME = "distilbert-base-uncased"   
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized = ds.map(tokenize_fn, batched=True, remove_columns=[c for c in ds["train"].column_names if c not in ["label_id"]])
tokenized = tokenized.rename_column("label_id", "labels")

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/7171 [00:00<?, ? examples/s]

Map:   0%|          | 0/797 [00:00<?, ? examples/s]

Map:   0%|          | 0/1992 [00:00<?, ? examples/s]

**TRAIN + METRICHE**

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABEL2ID),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    return {"accuracy": acc, "precision": p, "recall": r, "f1": f1}

args = TrainingArguments(
    output_dir="./clf_out",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_strategy="steps",
    logging_steps=50,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    tokenizer=tokenizer,             
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1388312774.py:32: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.015600,0.013945,0.996236,0.996265,0.996236,0.996236
2,0.000300,0.005600,0.998745,0.998748,0.998745,0.998745
3,0.000200,0.006718,0.997491,0.997504,0.997491,0.997491


TrainOutput(global_step=1347, training_loss=0.01970474147628481, metrics={'train_runtime': 106.9715, 'train_samples_per_second': 201.11, 'train_steps_per_second': 12.592, 'total_flos': 127190435642352.0, 'train_loss': 0.01970474147628481, 'epoch': 3.0})

**Evaluate + Save + Pipeline test**

In [ ]:
# --- evaluate sul test ---
test_metrics = trainer.evaluate(tokenized["test"])
print("TEST METRICS:", test_metrics)

# report dettagliato
pred = trainer.predict(tokenized["test"])
y_true = pred.label_ids
y_pred = np.argmax(pred.predictions, axis=-1)
print(classification_report(y_true, y_pred, target_names=[ID2LABEL[i] for i in range(len(ID2LABEL))]))

# --- save (modello + tokenizer + label map) ---
SAVE_DIR = "./model_classifier_en"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

with open(os.path.join(SAVE_DIR, "label_map.json"), "w") as f:
    json.dump(ID2LABEL, f, indent=2)

print("Salvato in:", SAVE_DIR)

# --- pipeline test (uso reale) ---
from transformers import pipeline

clf = pipeline("text-classification", model=SAVE_DIR, tokenizer=SAVE_DIR, device=0 if torch.cuda.is_available() else -1)

samples = [
    "Blonde women are usually not very smart.",
    "Hair color does not determine a person's intelligence."
]

for s in samples:
    print(s, "->", clf(s))


TEST METRICS: {'eval_loss': 0.012119488790631294, 'eval_accuracy': 0.9969879518072289, 'eval_precision': 0.996989955766664, 'eval_recall': 0.9969879518072289, 'eval_f1': 0.9969879487709359, 'eval_runtime': 0.6393, 'eval_samples_per_second': 3115.71, 'eval_steps_per_second': 98.539, 'epoch': 3.0}
               precision    recall  f1-score   support

    inclusive       1.00      1.00      1.00       996
not_inclusive       1.00      1.00      1.00       996

     accuracy                           1.00      1992
    macro avg       1.00      1.00      1.00      1992
 weighted avg       1.00      1.00      1.00      1992



Device set to use cuda:0


✅ Salvato in: ./model_classifier_en
Blonde women are usually not very smart. -> [{'label': 'not_inclusive', 'score': 0.9997733235359192}]
Hair color does not determine a person's intelligence. -> [{'label': 'inclusive', 'score': 0.9998733997344971}]
